# Denkwürfel: Stimme mit Qwen3-TTS

Erzeugt einen ganzen Sprechtext in deiner Stimme.

1. Oben **Laufzeit → Alle ausführen** (Strg+F9).
2. Bei der Warnung **Trotzdem ausführen** wählen.
3. Sobald der Knopf **Dateien auswählen** erscheint: `referenz-qwen.wav` auswählen.
4. Warten. Am Ende startet der Download von `sprechtext-qwen.wav`.
5. Danach **Laufzeit → Laufzeit trennen und löschen**. Damit ist deine Stimmprobe von der Colab-Maschine weg.

Dieses Notebook speichert keine Ausgaben, auch keine Audio-Player.


In [ ]:
# 1. Grafikkarte prüfen und Qwen installieren
!apt-get -qq install -y sox > /dev/null
!pip install -q qwen-tts==0.1.1

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "Keine Grafikkarte aktiv. Laufzeit → Laufzeittyp ändern → T4 GPU wählen, dann neu starten. "
        "Ist keine T4 frei, ist das Gratis-Kontingent gerade aufgebraucht: später erneut versuchen."
    )
torch.ones(1).to("cuda:0")
print("Grafikkarte:", torch.cuda.get_device_name(0))


In [ ]:
# 2. Text und Einstellungen
# Jeder Absatz wird einzeln erzeugt. Für ein neues Video nur TEXT austauschen.
TEXT = """
Dein Handy schickt für GPS kein einziges Signal ins All. Es hört nur zu.

Über dir kreisen gerade gut dreißig GPS-Satelliten. Jeder fliegt in rund zwanzigtausend Kilometern Höhe und umrundet die Erde zweimal am Tag.

Und jeder funkt ununterbrochen nur zwei Dinge: Wo bin ich? Und wie spät ist es bei mir? Gemessen mit Atomuhren an Bord.

Das Signal braucht bis zu dir etwa sieben Hundertstelsekunden. Dein Handy misst diese Verspätung. Und weil Funk mit Lichtgeschwindigkeit fliegt, wird aus Zeit eine Entfernung.

Eine Entfernung ergibt eine Kugel um den Satelliten. Drei Kugeln lassen nur einen Punkt auf der Erde übrig. Der vierte Satellit gleicht die ungenaue Uhr in deinem Handy aus. Und dieser Punkt bist du.

Jetzt der verrückte Teil: Da oben vergeht die Zeit schneller. Die Uhren der Satelliten gehen jeden Tag achtunddreißig Millionstelsekunden vor. Klingt nach nichts.

Aber ohne Einsteins Korrektur würde dein Standort jeden Tag um rund zehn Kilometer verrutschen. Während du dieses Video schaust, schon um mehrere Meter.

Deshalb werden die Uhren schon vor dem Start absichtlich ein winziges bisschen langsamer gestellt. Und dein Handy? Schickt dafür bis heute kein einziges Signal ins All.
"""

# So wird es gesprochen. Der längere Begriff muss zuerst stehen.
AUSSPRACHE = {
    "GPS-Satelliten": "Ge Pe Es Satelliten",
    "GPS": "Ge Pe Es",
}

# Wortlaut der Referenzaufnahme
REFERENZ_TEXT = "Es ist kurz nach neun Uhr morgens und am anderen Ende hörst du eine vertraute Stimme."

MODELL = "Qwen/Qwen3-TTS-12Hz-0.6B-Base"
SPRACHE = "German"
SEED = 7
MAX_TOKENS = 600  # 12 Tokens sind 1 Sekunde, also höchstens 50 Sekunden pro Absatz
PAUSE_SEKUNDEN = 0.3
ZIEL = "sprechtext-qwen.wav"


def sprechfassung(absatz):
    for geschrieben, gesprochen in AUSSPRACHE.items():
        absatz = absatz.replace(geschrieben, gesprochen)
    return absatz


absaetze = [sprechfassung(a.strip()) for a in TEXT.split("\n\n") if a.strip()]
print(len(absaetze), "Absätze,", sum(len(a.split()) for a in absaetze), "Wörter")


In [ ]:
# 3. Referenzaufnahme hochladen (referenz-qwen.wav)
import soundfile as sf
from google.colab import files

hochgeladen = files.upload()
wavs = [name for name in hochgeladen if name.lower().endswith(".wav")]
if len(wavs) != 1:
    raise RuntimeError("Bitte genau eine WAV-Datei hochladen: referenz-qwen.wav")
REFERENZ = wavs[0]
sekunden = sf.info(REFERENZ).duration
if not 3 <= sekunden <= 30:
    raise RuntimeError(f"Die Referenz ist {sekunden:.1f} s lang, erwartet sind 3 bis 30 s.")
print(f"Referenz: {REFERENZ}, {sekunden:.2f} s")


In [ ]:
# 4. Modell laden und Stimme vorbereiten (Download rund 2,4 GB, Lizenz Apache-2.0)
import time

from qwen_tts import Qwen3TTSModel

start = time.perf_counter()
modell = Qwen3TTSModel.from_pretrained(MODELL, device_map="cuda:0", dtype=torch.float32)
stimme = modell.create_voice_clone_prompt(ref_audio=REFERENZ, ref_text=REFERENZ_TEXT)
print(f"Modell bereit nach {time.perf_counter() - start:.0f} s")


In [ ]:
# 5. Alle Absätze erzeugen
import librosa
import numpy as np

teile = {}
abtastrate = None
rechenzeit = 0.0


def erzeuge(nr, seed=SEED):
    global abtastrate, rechenzeit
    torch.manual_seed(seed)
    start = time.perf_counter()
    wavs_neu, abtastrate = modell.generate_voice_clone(
        text=absaetze[nr - 1],
        language=SPRACHE,
        voice_clone_prompt=stimme,
        max_new_tokens=MAX_TOKENS,
    )
    dauer_rechnen = time.perf_counter() - start
    rechenzeit += dauer_rechnen
    ton = np.asarray(wavs_neu[0], dtype=np.float32)
    roh_sekunden = len(ton) / abtastrate
    teile[nr], _ = librosa.effects.trim(ton, top_db=45)
    print(f"Absatz {nr}: {roh_sekunden:.1f} s Audio in {dauer_rechnen:.0f} s")
    if roh_sekunden >= MAX_TOKENS / 12 * 0.98:
        print(f"  ACHTUNG: Absatz {nr} hat die Obergrenze erreicht und ist vermutlich falsch. Neu erzeugen.")


for nr in range(1, len(absaetze) + 1):
    erzeuge(nr)


In [ ]:
# 6. Zusammenfügen, anhören, herunterladen
from IPython.display import Audio, display


def fertigstellen():
    pause = np.zeros(int(PAUSE_SEKUNDEN * abtastrate), dtype=np.float32)
    stuecke = []
    for nr in sorted(teile):
        if stuecke:
            stuecke.append(pause)
        stuecke.append(teile[nr])
    gesamt = np.concatenate(stuecke)
    sf.write(ZIEL, gesamt, abtastrate, subtype="PCM_16")
    sekunden = len(gesamt) / abtastrate
    woerter = sum(len(a.split()) for a in absaetze)
    print(f"Fertig: {sekunden:.1f} s Audio, {woerter / sekunden * 60:.0f} Wörter pro Minute")
    print(f"Rechenzeit: {rechenzeit:.0f} s (Faktor {rechenzeit / sekunden:.1f})")
    display(Audio(ZIEL))
    files.download(ZIEL)


fertigstellen()


In [ ]:
# 7. Nur nötig, wenn ein Absatz falsch klingt
# Absatznummer und neuen Seed eintragen, z. B. NEU = {3: 8, 6: 9}, dann nur diese Zelle ausführen.
NEU = {}

for nr, seed in NEU.items():
    erzeuge(nr, seed=seed)
if NEU:
    fertigstellen()
